<a href="https://colab.research.google.com/github/stellar4554t/semiconductor-caculation/blob/main/Caculation_oxidation_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_oxidation(environment, T_celsius, t_hours, x_i):
    # Hằng số Boltzmann (eV/K)
    k = 8.617e-5
    T_kelvin = T_celsius + 273.15

    # Cấu hình thông số cho từng môi trường
    params = {
        'dry': {'C1': 7.72e2, 'E1': 1.23, 'C2': 6.23e6, 'E2': 2.0},
        'wet': {'C1': 2.14e2, 'E1': 0.71, 'C2': 8.95e7, 'E2': 2.05},
        'h2o': {'C1': 3.86e2, 'E1': 0.78, 'C2': 1.63e8, 'E2': 2.05}
    }

    p = params[environment.lower()]

    # 1. Tính B (Parabolic rate constant)
    B = p['C1'] * np.exp(-p['E1'] / (k * T_kelvin))

    # 2. Tính B/A (Linear rate constant) từ C2 và E2
    # Dựa trên công thức bạn đưa: A = B / (C2 * exp(-E2/kT)) => B/A = C2 * exp(-E2/kT)
    B_over_A = p['C2'] * np.exp(-p['E2'] / (k * T_kelvin))
    A = B / B_over_A

    # 3. Tính Tau (Thời gian hiệu chỉnh cho lớp oxit ban đầu)
    tau = (x_i**2 + A * x_i) / B

    # 4. Tính độ dày Oxit xo theo thời gian t
    # Công thức: xo = (A/2) * (sqrt(1 + (t + tau)/(A^2/4B)) - 1)
    xo = (A / 2) * (np.sqrt(1 + (t_hours + tau) / (A**2 / (4 * B))) - 1)

    return xo

# Lấy input từ người dùng cho x_i
x_i = float(input("độ dày lớp oxide ban đầu (µm): "))

# --- THIẾT LẬP MÔ PHỎNG ---
t_range = np.linspace(0, 10, 100)  # Thời gian từ 0 đến 10 giờ
temperatures = [700, 800, 900, 1000, 1100, 1200]
environments = ['dry', 'wet', 'h2o']

# Vẽ biểu đồ cho mỗi môi trường
for current_env in environments:
    plt.figure(figsize=(10, 6))
    for T in temperatures:
        thickness = [simulate_oxidation(current_env, T, t, x_i=x_i) for t in t_range]
        plt.plot(t_range, thickness, label=f'T = {T}°C')

    plt.title(f'Sự phát triển lớp SiO2 trong môi trường {current_env.upper()} (x_i={x_i:.2f} µm)')
    plt.xlabel('Thời gian (giờ)')
    plt.ylabel('Độ dày Oxit (µm)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import os

# ==================================================
# OUTPUT
# ==================================================
OUTDIR = "tcad_scaled_TP"
os.makedirs(OUTDIR, exist_ok=True)

# ==================================================
# GRID
# ==================================================
x_max, y_max = 0.25, 1.0
nx, ny = 200, 300

dx = x_max / nx
dy = y_max / ny

x = np.linspace(0, x_max, nx)
y = np.linspace(0, y_max, ny)

# ==================================================
# TIME
# ==================================================
dt = 1e-4
t_end = 0.15
time = np.arange(0, t_end, dt)

# ==================================================
# USER INPUT: TEMPERATURE & PRESSURE
# ==================================================
T_C = 1000                 # °C
P = 1.0                    # atm (try 0.5, 1.0, 2.0)

T_ref = 1000 + 273.15
P_ref = 1.0

T = T_C + 273.15

# ==================================================
# DOPANT PROFILE
# ==================================================
N0 = 5e19
lambda_x = 0.04
lambda_y = 0.15

X, Y = np.meshgrid(x, y, indexing="ij")
N = N0 * np.exp(-X/lambda_x) * np.exp(-((Y-y_max/2)**2)/lambda_y**2)

# ==================================================
# DEAL–GROVE (SCALED BASE)
# ==================================================
A0 = 0.08       # µm
B0 = 0.012      # µm^2/hour

# Temperature scaling
A_T = A0 * (T / T_ref)
B_T = B0 * (T / T_ref)**2

# Pressure scaling
A_TP = A_T * (P / P_ref)
B_TP = B_T * (P / P_ref)**2

# ==================================================
# DOPANT PARAMETERS
# ==================================================
alpha = 2.0
m = 1
Nref = 1e19
k_seg = 1.5

# ==================================================
# FUNCTION: RUN ONE CASE
# ==================================================
def run_simulation(use_dopant, tag):

    x_ox = np.ones(ny) * 0.002

    fig, ax = plt.subplots(figsize=(7, 4))

    im = ax.imshow(
        N if use_dopant else np.zeros_like(N),
        extent=[0, y_max, x_max, 0],
        aspect="auto",
        cmap="inferno",
        vmin=0,
        vmax=N0
    )

    line, = ax.plot(y, x_ox, "c", lw=2)
    time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes, color="white")

    title = "WITH Dopant" if use_dopant else "WITHOUT Dopant"
    ax.set_title(f"Thermal Oxidation ({title}, {T_C}°C, {P} atm)")
    ax.set_xlabel("Lateral y (µm)")
    ax.set_ylabel("Depth x (µm)")
    plt.colorbar(im, ax=ax, label="Dopant concentration (cm$^{-3}$)")

    def update(frame):
        nonlocal x_ox

        t = time[frame]

        if use_dopant:
            i_int = np.clip((x_ox / dx).astype(int), 1, nx-2)
            N_int = k_seg * N[i_int, np.arange(ny)]
            A_eff = A_TP * (1 + alpha * (N_int/Nref)**m)
        else:
            A_eff = A_TP

        v = B_TP / (2*x_ox + A_eff)
        x_ox += v * dt

        line.set_ydata(x_ox)
        time_text.set_text(f"t = {t*60:.2f} min")

        return line, time_text

    ani = animation.FuncAnimation(
        fig, update,
        frames=len(time),
        interval=40,
        blit=True
    )

    gif_path = os.path.join(OUTDIR, f"oxidation_{tag}.gif")
    ani.save(gif_path, writer="pillow", fps=20)
    plt.close()

    print("Saved:", gif_path)

# ==================================================
# RUN BOTH CASES
# ==================================================
run_simulation(False, "no_dopant")
run_simulation(True,  "with_dopant")

print("\nSimulation finished (Scaled TCAD with Temperature + Pressure)")
